# Config

In [ ]:
import requests
from kafka import KafkaProducer
import json
import os
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
BASE_URL = "https://prim.iledefrance-mobilites.fr/marketplace/v2/navitia"
PRIM_API_KEY = os.getenv("PRIM_API_KEY")
kafka_config = {
        'bootstrap_servers': 'kafka1:9092',  # Update with your Kafka broker
        'serializer': lambda v: json.dumps(v).encode('utf-8')  # Serialize data to JSON
}

# Producer

In [ ]:
class Producer:
    def __init__(self, config, auth_token:str, base_url:str):
        self.producer = KafkaProducer(
            bootstrap_servers=config['bootstrap_servers'],
            value_serializer=config['serializer']
        )
        self.auth_token = auth_token
        self.base_url = base_url
        self.start_time = datetime.now().strftime("%Y%m%dT%H%M%S")

    def get(self, url, params):
        return requests.get(
            url, 
            params=params, 
            headers={"apiKey": self.auth_token}
        )
    
    def send_to_kafka(self, topics, endpoint, params):
        response = self.get(f"{self.base_url}/{endpoint}", params)
        if response.status_code == 200:
            data = response.json()
            for topic in topics:
                self.producer.send(topic, data[topic])
                self.producer.flush()
            print(f"Page {data['start_page']} of data uploaded to kafka")
            return data['total_result']
        else:
            print(f"Failed to fetch data: {response.status_code}")
        
    
    def ingest_line_reports(self, count=100):
        params = {
            "since": self.start_time,
            "count": count
        }
        topics = [
            "line_reports",
            "links",
            "disruptions"
        ]
        total_result = self.send_to_kafka(topics, "line_reports", params)
        for i in range(1, total_result//count+1):
            params['start_page'] = i
            self.send_to_kafka(topics, "line_reports", params)

# Start data collection

In [ ]:
producer = Producer(kafka_config, PRIM_API_KEY, BASE_URL)
producer.ingest_line_reports()